# Remote Browser Download Example

This notebook demonstrates downloading files from a remote browser with agent awareness.
The agent can monitor download progress and make intelligent decisions.

## Features Demonstrated
- Remote browser file downloads using HTTP client
- Agent awareness of download progress

## Setup and Imports

In [ ]:
import asyncio
from contextlib import suppress
from pathlib import Path

from boto3.session import Session
from rich.console import Console

from bedrock_agentcore.tools.browser_client import BrowserClient
from browser_use import Agent, Browser, BrowserProfile
from browser_use.llm import ChatAnthropicBedrock

console = Console()

## Initialize AWS Session and Browser Client

In [ ]:
# Initialize AWS session
boto_session = Session()
region = boto_session.region_name
print(f"Using AWS region: {region}")

# Create browser client
client = BrowserClient(region)
client.start()

# Get WebSocket URL and headers for remote browser
ws_url, headers = client.generate_ws_headers()
console.print("✅ Remote browser client initialized", style="green")

## Configure Browser Profile for Remote Downloads

In [ ]:
# Create downloads directory
downloads_dir = Path("./downloads")
downloads_dir.mkdir(exist_ok=True)

# Configure browser profile with remote download support
browser_profile = BrowserProfile(
    headers=headers,
    timeout=1500000,
    downloads_path=str(downloads_dir),
    download_from_remote_browser=True,  # Enable HTTP downloads for remote browser
    auto_download_pdfs=True
)

console.print(f"📁 Downloads will be saved to: {downloads_dir.absolute()}", style="blue")

## Start Browser Session

In [ ]:
# Create and start browser session
browser_session = Browser(
    cdp_url=ws_url,
    browser_profile=browser_profile,
    keep_alive=True
)

await browser_session.start()
console.print("✅ Remote browser session started", style="green")

## Initialize LLM and Create Agent

In [ ]:
# Initialize Bedrock LLM
bedrock_chat = ChatAnthropicBedrock(
    model='us.anthropic.claude-3-7-sonnet-20250219-v1:0',
    aws_region='us-west-2'
)

# Define task with download monitoring
task = (
    "Go to https://proof.ovh.net/files and download the 100 MB file. "
    "Monitor the download progress and tell me when it's complete. "
    "If the download fails, try again."
)

# Create agent with download awareness
agent = Agent(
    task=task,
    llm=bedrock_chat,
    browser_session=browser_session,
    llm_timeout=300
)

console.print("🤖 Agent created with download monitoring capabilities", style="blue")

## Run Download Task

In [ ]:
try:
    console.print("🚀 Starting download task...", style="blue")
    result = await agent.run()
    
    console.print("✅ Task completed!", style="green")
    print(f"\nAgent result: {result}")
    
except Exception as e:
    console.print(f"❌ Error during task execution: {e}", style="red")
    raise

## Cleanup Resources

In [ ]:
# Clean up browser session and client
try:
    if browser_session:
        await browser_session.close()
        console.print("🔄 Browser session closed", style="yellow")
except Exception as e:
    console.print(f"⚠️ Error closing browser session: {e}", style="yellow")

try:
    client.stop()
    console.print("🔄 Browser client stopped", style="yellow")
except Exception as e:
    console.print(f"⚠️ Error stopping client: {e}", style="yellow")

console.print("✅ Cleanup completed", style="green")

## Summary

This notebook demonstrated:

1. **Remote Browser Setup** - Connected to a remote browser instance
2. **Download Configuration** - Enabled `download_from_remote_browser=True` for HTTP-based file transfers
3. **Agent Awareness** - The agent monitored download progress and made intelligent decisions
4. **File Transfer** - Successfully downloaded files from remote browser to local machine
5. **Resource Management** - Proper cleanup of browser sessions and clients

### Key Features Used:
- `download_from_remote_browser=True` - Enables HTTP client downloads
- `auto_download_pdfs=True` - Automatic PDF download detection
- Agent context awareness - Download progress visible to LLM for decision making

This approach works seamlessly in containerized environments, cloud deployments, and any scenario where the browser runs remotely from your local machine.